# 05 feature engineering

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
# Load session ground truth and attributed behavioural events

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = PUBLICATION_ROOT
OUTPUT_DIR = PROJECT_ROOT / "data" / "notebook_outputs"

# Load ground-truth dataset
ground_truth = pd.read_csv(
    OUTPUT_DIR / "session_ground_truth.csv"
)

# Load events after orchestration artefact exclusion
events = pd.read_csv(
    OUTPUT_DIR / "wazuh_attributed_events_filtered.csv",
    dtype=str
)

# Create one row for every simulated session
session_features = ground_truth[["SessionID"]].copy()

print("Total sessions:", len(session_features))
print("Behavioural events:", len(events))

display(session_features.head())

In [ ]:
# Calculate basic behavioural event counts per session

SECURITY = "Microsoft-Windows-Security-Auditing"
SYSMON = "Microsoft-Windows-Sysmon"

security = events["Provider"].eq(SECURITY)
sysmon = events["Provider"].eq(SYSMON)
event_id = events["EventID"]

# Total attributed behavioural events per session
total_counts = events.groupby("SessionID").size()

session_features["windows_event_count"] = (
    session_features["SessionID"]
    .map(total_counts)
    .fillna(0)
    .astype(int)
)

# Define event-specific behavioural features
feature_conditions = {
    "failed_logon_count":
        security & event_id.eq("4625"),

    "successful_logon_count":
        security & event_id.eq("4624"),

    "smb_access_check_count":
        security & event_id.eq("5145"),

    "file_access_count":
        security
        & event_id.eq("4663")
        & events["ObjectType"].fillna("").str.casefold().eq("file"),

    "object_deletion_event_count":
        security & event_id.eq("4660"),

    "file_create_event_count":
        sysmon & event_id.eq("11"),

    "sysmon_file_delete_count":
        sysmon & event_id.isin(["23", "26"]),

    "process_create_count":
        sysmon & event_id.eq("1"),

    "network_connection_count":
        sysmon & event_id.eq("3")
}

# Aggregate each feature by SessionID
for feature, condition in feature_conditions.items():

    counts = events.loc[condition].groupby("SessionID").size()

    session_features[feature] = (
        session_features["SessionID"]
        .map(counts)
        .fillna(0)
        .astype(int)
    )

print("Sessions:", len(session_features))
print("Features calculated:", len(session_features.columns) - 1)

display(session_features.head())

In [ ]:
# Calculate credential, SMB denial and file access features

import pandas as pd

def parse_windows_integer(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    try:
        return int(value, 0)
    except ValueError:
        try:
            return int(value, 10)
        except ValueError:
            return None


def add_count_feature(name, selected_events):
    counts = selected_events.groupby("SessionID").size()

    session_features[name] = (
        session_features["SessionID"]
        .map(counts)
        .fillna(0)
        .astype("Int64")
    )


def mark_unreliable(name, affected_sessions):
    session_features.loc[
        session_features["SessionID"].isin(affected_sessions),
        name
    ] = pd.NA


# 1. Credential validation failures (Security 4776)

credential_events = events.loc[
    security & event_id.eq("4776")
].copy()

credential_events["ParsedStatus"] = (
    credential_events["Status"].map(parse_windows_integer)
)

add_count_feature(
    "credential_validation_failure_count",
    credential_events.loc[
        credential_events["ParsedStatus"].notna()
        & credential_events["ParsedStatus"].ne(0)
    ]
)

mark_unreliable(
    "credential_validation_failure_count",
    credential_events.loc[
        credential_events["ParsedStatus"].isna(),
        "SessionID"
    ]
)


# 2. SMB access denials (Security 5145)

def is_audit_failure(row):
    keywords = parse_windows_integer(row["Keywords"])
    severity = row["SeverityValue"]

    failure = (
        bool(keywords & 0x0010000000000000)
        if keywords is not None else False
    )

    success = (
        bool(keywords & 0x0020000000000000)
        if keywords is not None else False
    )

    if failure and not success and severity != "AUDIT_SUCCESS":
        return True

    if success and not failure and severity != "AUDIT_FAILURE":
        return False

    if not (failure or success):
        if severity == "AUDIT_FAILURE":
            return True
        if severity == "AUDIT_SUCCESS":
            return False

    return None


smb_events = events.loc[
    security & event_id.eq("5145")
].copy()

smb_events["AuditFailure"] = smb_events.apply(
    is_audit_failure,
    axis=1
)

add_count_feature(
    "smb_denied_count",
    smb_events.loc[smb_events["AuditFailure"].eq(True)]
)

mark_unreliable(
    "smb_denied_count",
    smb_events.loc[
        smb_events["AuditFailure"].isna(),
        "SessionID"
    ]
)


# 3. File read, write and DELETE access rights (Security 4663)

file_events = events.loc[
    security
    & event_id.eq("4663")
    & events["ObjectType"].fillna("").str.casefold().eq("file")
].copy()

file_events["ParsedMask"] = (
    file_events["AccessMask"].map(parse_windows_integer)
)

invalid_mask_sessions = file_events.loc[
    file_events["ParsedMask"].isna(),
    "SessionID"
]

valid_file_events = file_events.loc[
    file_events["ParsedMask"].notna()
]

access_features = {
    "file_read_access_count": 0x1,
    "file_write_access_count": 0x6,
    "delete_access_count": 0x10000
}

for feature, mask in access_features.items():

    matching = valid_file_events.loc[
        valid_file_events["ParsedMask"].map(
            lambda value: bool(value & mask)
        )
    ]

    add_count_feature(feature, matching)

    mark_unreliable(feature, invalid_mask_sessions)


print("Sessions:", len(session_features))
print("Features calculated:", len(session_features.columns) - 1)

display(session_features.head())

In [ ]:
# Calculate process and network behavioural features

import ntpath

# 1. Process-related features (Sysmon Event 1)

process_events = events.loc[
    sysmon & event_id.eq("1")
].copy()

# Extract executable filename
process_events["ImageName"] = (
    process_events["Image"]
    .fillna("")
    .map(lambda value: ntpath.basename(value).casefold())
)

# SSH process executions
add_count_feature(
    "ssh_process_count",
    process_events.loc[
        process_events["ImageName"].eq("ssh.exe")
    ]
)

# SCP process executions
add_count_feature(
    "scp_process_count",
    process_events.loc[
        process_events["ImageName"].isin(["scp.exe", "pscp.exe"])
    ]
)

# Missing executable names make these features unreliable
missing_image_sessions = process_events.loc[
    process_events["ImageName"].eq(""),
    "SessionID"
]

for feature in ["ssh_process_count", "scp_process_count"]:
    mark_unreliable(feature, missing_image_sessions)


# 2. Distinct process GUIDs

process_events["NormalizedGuid"] = (
    process_events["ProcessGuid"].fillna("").str.casefold()
)

process_events["NormalizedHost"] = (
    process_events["Computer"].fillna("").str.casefold()
)

valid_processes = process_events.loc[
    process_events["NormalizedGuid"].ne("")
]

distinct_processes = (
    valid_processes
    .drop_duplicates([
        "SessionID",
        "NormalizedHost",
        "NormalizedGuid"
    ])
    .groupby("SessionID")
    .size()
)

session_features["distinct_process_guids"] = (
    session_features["SessionID"]
    .map(distinct_processes)
    .fillna(0)
    .astype("Int64")
)

mark_unreliable(
    "distinct_process_guids",
    process_events.loc[
        process_events["NormalizedGuid"].eq(""),
        "SessionID"
    ]
)


# 3. Network-related features (Sysmon Event 3)

network_events = events.loc[
    sysmon & event_id.eq("3")
].copy()

# Count distinct destination IP addresses
valid_destinations = network_events.loc[
    network_events["DestinationIp"].notna()
    & network_events["DestinationIp"].ne("")
]

distinct_destinations = (
    valid_destinations
    .drop_duplicates(["SessionID", "DestinationIp"])
    .groupby("SessionID")
    .size()
)

session_features["distinct_network_destinations"] = (
    session_features["SessionID"]
    .map(distinct_destinations)
    .fillna(0)
    .astype("Int64")
)

mark_unreliable(
    "distinct_network_destinations",
    network_events.loc[
        network_events["DestinationIp"].isna()
        | network_events["DestinationIp"].eq(""),
        "SessionID"
    ]
)


# 4. Initiated SSH connections to port 22

network_events["ParsedPort"] = (
    network_events["DestinationPort"]
    .map(parse_windows_integer)
)

network_events["InitiatedValue"] = (
    network_events["Initiated"]
    .fillna("")
    .str.casefold()
)

valid_network_records = (
    network_events["ParsedPort"].notna()
    & network_events["InitiatedValue"].isin(["true", "false"])
)

ssh_connections = network_events.loc[
    valid_network_records
    & network_events["ParsedPort"].eq(22)
    & network_events["InitiatedValue"].eq("true")
]

add_count_feature(
    "ssh_port22_connection_count",
    ssh_connections
)

mark_unreliable(
    "ssh_port22_connection_count",
    network_events.loc[
        ~valid_network_records,
        "SessionID"
    ]
)

print("Sessions:", len(session_features))
print("Features calculated:", len(session_features.columns) - 1)

display(session_features.head())

In [ ]:
# Calculate SFTP process and file-path behavioural features

import re
import ntpath
import pandas as pd

SFTP_IMAGES = {
    "sftp.exe", "psftp.exe", "winscp.exe", "winscp.com"
}

# 1. SFTP process creation (Sysmon Event 1)

add_count_feature(
    "sftp_process_count",
    process_events.loc[
        process_events["ImageName"].isin(SFTP_IMAGES)
    ]
)

mark_unreliable(
    "sftp_process_count",
    missing_image_sessions
)


# 2. Normalise Windows file paths

def normalize_windows_path(value):
    if not isinstance(value, str) or not value:
        return ""

    value = value.replace("/", "\\").casefold()
    unc = value.startswith("\\\\")

    # Collapse repeated separators
    value = re.sub(r"\\+", lambda _: "\\", value)

    # Remove Windows extended path prefixes
    if value.startswith("\\??\\"):
        value = value[4:]
    elif value.startswith("\\?\\"):
        value = value[3:]
    elif unc and value.startswith("\\"):
        value = "\\" + value

    return ntpath.normpath(value)


def get_event_resource_path(row):
    provider = row["Provider"]
    event_id = row["EventID"]

    if provider == SECURITY and event_id == "5145":
        base = row["ShareLocalPath"] or row["ShareName"]
        relative = row["RelativeTargetName"]

        if base and relative:
            return normalize_windows_path(
                ntpath.join(
                    normalize_windows_path(base),
                    relative.lstrip("\\/")
                )
            )

        return ""

    if (
        provider == SECURITY
        and event_id == "4663"
        and row["ObjectType"].casefold() == "file"
    ):
        return normalize_windows_path(row["ObjectName"])

    if provider == SYSMON and event_id in {"11", "23", "26"}:
        return normalize_windows_path(row["TargetFilename"])

    return ""


# 3. Calculate resource paths for attributed events

path_events = events.fillna("").copy()

path_events["ResourcePath"] = path_events.apply(
    get_event_resource_path,
    axis=1
)

path_events["NormalizedHost"] = (
    path_events["Computer"].str.casefold()
)

# Events belonging to the reference's file family
file_family = (
    (
        path_events["Provider"].eq(SECURITY)
        & (
            path_events["EventID"].eq("5145")
            | (
                path_events["EventID"].eq("4663")
                & path_events["ObjectType"].str.casefold().eq("file")
            )
        )
    )
    | (
        path_events["Provider"].eq(SYSMON)
        & path_events["EventID"].isin(["11", "23", "26"])
    )
)

smb_family = (
    path_events["Provider"].eq(SECURITY)
    & path_events["EventID"].eq("5145")
)


# 4. Count distinct file and SMB paths

def add_distinct_path_feature(name, family_mask):
    selected = path_events.loc[family_mask]

    valid = selected.loc[
        selected["ResourcePath"].ne("")
    ]

    distinct_counts = (
        valid
        .drop_duplicates([
            "SessionID",
            "NormalizedHost",
            "ResourcePath"
        ])
        .groupby("SessionID")
        .size()
    )

    session_features[name] = (
        session_features["SessionID"]
        .map(distinct_counts)
        .fillna(0)
        .astype("Int64")
    )

    # Missing file paths make the distinct count unreliable
    mark_unreliable(
        name,
        selected.loc[
            selected["ResourcePath"].eq(""),
            "SessionID"
        ]
    )


add_distinct_path_feature(
    "distinct_file_paths",
    file_family
)

add_distinct_path_feature(
    "distinct_smb_paths",
    smb_family
)


# 5. Count activity involving protected resources
# Retain for diagnostics, but exclude from ML predictors

PROTECTED_PREFIXES = ("restricted",)

def is_protected_path(path):
    if not path:
        return False

    return any(
        component.startswith(PROTECTED_PREFIXES)
        for component in path.split("\\")
    )


protected = path_events.loc[
    path_events["ResourcePath"].map(is_protected_path)
]

add_count_feature(
    "protected_resource_event_count",
    protected
)

mark_unreliable(
    "protected_resource_event_count",
    path_events.loc[
        file_family & path_events["ResourcePath"].eq(""),
        "SessionID"
    ]
)

print("Sessions:", len(session_features))
print("Features calculated:", len(session_features.columns) - 1)

display(session_features.head())

In [ ]:
# Load process evidence generated in Notebook 04

import pickle

process_file = OUTPUT_DIR / "process_history.pkl"

with open(process_file, "rb") as f:
    process_history = pickle.load(f)

print("Process evidence loaded.")
print("Unique processes:", len(process_history))

In [ ]:
# Calculate unique action signatures and SFTP network connections

import ntpath
import pandas as pd

# --------------------------------------------------
# 1. Unique action signatures
# --------------------------------------------------

def get_field(row, column):
    value = row.get(column, "")
    return "" if pd.isna(value) else str(value)


def get_fallback_field(row, primary, fallback):
    return get_field(row, primary) or get_field(row, fallback)


def create_action_signature(row):
    return (
        get_field(row, "Computer").casefold(),
        get_field(row, "Provider"),
        get_field(row, "EventID"),
        get_field(row, "ResolvedUsername"),
        get_field(row, "ResourcePath"),
        get_field(row, "AccessMask"),
        get_field(row, "HandleId"),
        get_fallback_field(row, "SubjectLogonId", "LogonId"),
        get_fallback_field(row, "ProcessGuid", "ProcessId"),
        ntpath.basename(get_field(row, "Image")).casefold(),
        get_field(row, "DestinationIp"),
        get_field(row, "DestinationPort")
    )


path_events["ActionSignature"] = path_events.apply(
    create_action_signature,
    axis=1
)

signature_counts = (
    path_events
    .drop_duplicates(["SessionID", "ActionSignature"])
    .groupby("SessionID")
    .size()
)

session_features["unique_action_signatures"] = (
    session_features["SessionID"]
    .map(signature_counts)
    .fillna(0)
    .astype("Int64")
)


# --------------------------------------------------
# 2. SFTP network connections (Sysmon Event 3)
# --------------------------------------------------

SFTP_IMAGES = {
    "sftp.exe",
    "psftp.exe",
    "winscp.exe",
    "winscp.com"
}


def get_process_evidence(host, guid, event_time, available_time):
    if (
        not isinstance(host, str)
        or not isinstance(guid, str)
        or not guid
        or pd.isna(event_time)
        or pd.isna(available_time)
    ):
        return None

    history = process_history.get(
        (host.casefold(), guid.casefold()),
        []
    )

    evidence = [
        item for item in history
        if item["AvailableTime"] <= available_time
    ]

    starts = {
        item["EventTime"]
        for item in evidence
        if item["EventID"] == "1"
        and item["EventTime"] <= event_time
    }

    identities = {
        item["Identity"]
        for item in evidence
        if isinstance(item["Identity"], str)
        and item["Identity"]
    }

    creation_events = [
        item for item in evidence
        if item["EventID"] == "1"
        and item["EventTime"] <= event_time
    ]

    images = {
        ntpath.basename(item["Image"]).casefold()
        for item in creation_events
        if isinstance(item["Image"], str)
    }

    parents = {
        item["ParentProcessGuid"].casefold()
        if isinstance(item["ParentProcessGuid"], str)
        else ""
        for item in creation_events
    }

    ends = {
        item["EventTime"]
        for item in evidence
        if item["EventID"] == "5"
    }

    if (
        len(starts) != 1
        or len(identities) != 1
        or len(images) != 1
    ):
        return None

    if ends and event_time > min(ends):
        return None

    return {
        "Image": next(iter(images)),
        "Parents": parents
    }


def is_sftp_network_connection(row):
    image = ntpath.basename(
        get_field(row, "Image")
    ).casefold()

    # Direct SFTP process network activity
    if image in SFTP_IMAGES:
        return True

    # Only SSH processes can qualify through their parent
    if image != "ssh.exe":
        return False

    host = row["Computer"]
    guid = row["ProcessGuid"]
    event_time = row["EventTimeUTC"]
    available_time = row["AvailableTimeUTC"]

    process = get_process_evidence(
        host, guid, event_time, available_time
    )

    if not process or len(process["Parents"]) != 1:
        return False

    parent_guid = next(iter(process["Parents"]))

    parent = get_process_evidence(
        host, parent_guid, event_time, available_time
    )

    return bool(
        parent and parent["Image"] in SFTP_IMAGES
    )


network_events["EventTimeUTC"] = pd.to_datetime(
    network_events["EventTimeUTC"],
    utc=True,
    errors="coerce"
)

network_events["AvailableTimeUTC"] = pd.to_datetime(
    network_events["AvailableTimeUTC"],
    utc=True,
    errors="coerce"
)

network_events["IsSftpNetwork"] = network_events.apply(
    is_sftp_network_connection,
    axis=1
)

add_count_feature(
    "sftp_network_connection_count",
    network_events.loc[network_events["IsSftpNetwork"]]
)

# Missing executable names make SFTP classification unreliable
mark_unreliable(
    "sftp_network_connection_count",
    network_events.loc[
        network_events["Image"].fillna("").eq(""),
        "SessionID"
    ]
)


# --------------------------------------------------
# 3. Display final feature count
# --------------------------------------------------

print("Total sessions:", len(session_features))
print("Behavioural features:", len(session_features.columns) - 1)

display(
    session_features[
        [
            "SessionID",
            "unique_action_signatures",
            "sftp_process_count",
            "sftp_network_connection_count"
        ]
    ].head(10)
)

In [ ]:
# Compare Jupyter feature calculations with Codex reference

reference_file = (
    PROJECT_ROOT / "data" / "processed" / "session_features.csv"
)

reference = pd.read_csv(reference_file)

# Select the 26 behavioural features
feature_columns = [
    col for col in session_features.columns
    if col != "SessionID"
]

# Match records using SessionID
comparison = session_features.merge(
    reference[["SessionID"] + feature_columns],
    on="SessionID",
    how="outer",
    suffixes=("_jupyter", "_codex"),
    validate="one_to_one",
    indicator=True
)

results = []

for feature in feature_columns:

    jupyter_values = pd.to_numeric(
        comparison[f"{feature}_jupyter"],
        errors="coerce"
    )

    codex_values = pd.to_numeric(
        comparison[f"{feature}_codex"],
        errors="coerce"
    )

    # Treat two missing values as matching
    matches = (
        jupyter_values.eq(codex_values).fillna(False)
        | (jupyter_values.isna() & codex_values.isna())
    )

    results.append({
        "Feature": feature,
        "MatchingSessions": int(matches.sum()),
        "MismatchedSessions": int((~matches).sum())
    })

comparison_summary = pd.DataFrame(results)

comparison_summary = comparison_summary.sort_values(
    "MismatchedSessions",
    ascending=False
)

print("Total sessions compared:", len(comparison))
print("Total features compared:", len(feature_columns))

print(
    "Features with exact agreement:",
    int((comparison_summary["MismatchedSessions"] == 0).sum())
)

print(
    "Features with differences:",
    int((comparison_summary["MismatchedSessions"] > 0).sum())
)

print("\nSession alignment:")
print(comparison["_merge"].value_counts())

display(comparison_summary)

In [ ]:
# Investigate the two mismatched features

features_to_check = {
    "sysmon_file_delete_count": "Observed_sysmon_delete",
    "object_deletion_event_count": "Observed_deletion"
}

for feature, observation_flag in features_to_check.items():

    jupyter_col = f"{feature}_jupyter"
    codex_col = f"{feature}_codex"

    jupyter_values = pd.to_numeric(
        comparison[jupyter_col], errors="coerce"
    )

    codex_values = pd.to_numeric(
        comparison[codex_col], errors="coerce"
    )

    mismatches = ~(
        jupyter_values.eq(codex_values).fillna(False)
        | (jupyter_values.isna() & codex_values.isna())
    )

    print(f"\nFeature: {feature}")
    print("Jupyter missing values:", jupyter_values.isna().sum())
    print("Codex missing values:", codex_values.isna().sum())
    print("Mismatched sessions:", mismatches.sum())

    examples = comparison.loc[
        mismatches,
        ["SessionID", jupyter_col, codex_col]
    ].head(10).copy()

    examples = examples.merge(
        reference[["SessionID", observation_flag]],
        on="SessionID",
        how="left"
    )

    display(examples)

In [ ]:
# Reconstruct deletion telemetry observability from Wazuh events

import pandas as pd

SECURITY = "Microsoft-Windows-Security-Auditing"
SYSMON = "Microsoft-Windows-Sysmon"

# 1. Load session prediction cutoffs

cutoffs = pd.read_csv(
    OUTPUT_DIR / "session_prediction_cutoffs.csv",
    dtype=str
)

cutoffs["PredictionCutoffUTC"] = pd.to_datetime(
    cutoffs["PredictionCutoffUTC"],
    utc=True,
    errors="coerce",
    format="mixed"
)


# 2. Identify hosts and dates associated with each session
# Use all attributed events, including orchestration events,
# because the reference records host/day locations before filtering.

attributed = pd.read_csv(
    OUTPUT_DIR / "wazuh_attributed_events.csv",
    usecols=["SessionID", "Computer", "WindowsEventTime"],
    dtype=str
)

attributed["EventUTC"] = pd.to_datetime(
    attributed["WindowsEventTime"],
    utc=True,
    errors="coerce",
    format="mixed"
)

attributed["Host"] = (
    attributed["Computer"].fillna("").str.casefold()
)

attributed["Day"] = (
    attributed["EventUTC"].dt.strftime("%Y-%m-%d")
)

locations = (
    attributed.loc[
        attributed["EventUTC"].notna()
        & attributed["Host"].ne(""),
        ["SessionID", "Host", "Day"]
    ]
    .drop_duplicates()
    .merge(
        cutoffs,
        on="SessionID",
        how="left",
        validate="many_to_one"
    )
)


# 3. Build host/day support evidence from Wazuh telemetry
# Use pre-deduplication events to preserve earliest receipt evidence.

source_file = OUTPUT_DIR / "wazuh_relevant_events.csv"

first_seen = {}

for chunk in pd.read_csv(
    source_file,
    usecols=[
        "Provider",
        "EventChannel",
        "EventID",
        "Computer",
        "WindowsEventTime",
        "WazuhTimestamp"
    ],
    chunksize=50000,
    dtype=str
):

    deletion = (
        chunk["Provider"].eq(SECURITY)
        & chunk["EventChannel"].eq("Security")
        & chunk["EventID"].eq("4660")
    )

    sysmon_delete = (
        chunk["Provider"].eq(SYSMON)
        & chunk["EventChannel"].eq(
            "Microsoft-Windows-Sysmon/Operational"
        )
        & chunk["EventID"].isin(["23", "26"])
    )

    selected = chunk.loc[deletion | sysmon_delete].copy()

    if selected.empty:
        continue

    selected["Family"] = "sysmon_delete"

    selected.loc[
        selected["Provider"].eq(SECURITY),
        "Family"
    ] = "deletion"

    selected["EventUTC"] = pd.to_datetime(
        selected["WindowsEventTime"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    selected["ReceiptUTC"] = pd.to_datetime(
        selected["WazuhTimestamp"],
        utc=True,
        errors="coerce",
        format="mixed"
    )

    selected["Host"] = (
        selected["Computer"].fillna("").str.casefold()
    )

    selected = selected.dropna(
        subset=["EventUTC", "ReceiptUTC"]
    )

    selected = selected.loc[
        selected["Host"].ne("")
    ].copy()

    if selected.empty:
        continue

    selected["AvailableUTC"] = selected[
        ["EventUTC", "ReceiptUTC"]
    ].max(axis=1)

    selected["Day"] = (
        selected["EventUTC"].dt.strftime("%Y-%m-%d")
    )

    earliest = selected.groupby(
        ["Family", "Host", "Day"]
    )["AvailableUTC"].min()

    for key, ready_time in earliest.items():
        previous = first_seen.get(key)

        if previous is None or ready_time < previous:
            first_seen[key] = ready_time


# 4. Determine whether each deletion family was observable
# before the session's prediction cutoff

feature_families = {
    "object_deletion_event_count": "deletion",
    "sysmon_file_delete_count": "sysmon_delete"
}

for feature, family in feature_families.items():

    support_times = pd.Series(
        pd.to_datetime(
            [
                first_seen.get(
                    (family, host, day),
                    pd.NaT
                )
                for host, day in zip(
                    locations["Host"],
                    locations["Day"]
                )
            ],
            utc=True,
            errors="coerce"
        ),
        index=locations.index
    )

    observable = (
        support_times.notna()
        & (
            support_times
            <= locations["PredictionCutoffUTC"]
        )
    )

    observed_sessions = set(
        locations.loc[observable, "SessionID"]
    )

    # Preserve existing counts, but mark unavailable
    # telemetry as missing rather than zero.
    session_features[feature] = (
        session_features[feature].astype("Int64")
    )

    session_features.loc[
        ~session_features["SessionID"].isin(observed_sessions),
        feature
    ] = pd.NA

    print(f"\n{feature}")
    print("Observable sessions:", len(observed_sessions))
    print(
        "Missing values:",
        session_features[feature].isna().sum()
    )


# 5. Compare the corrected features with Codex

reference = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "session_features.csv",
    usecols=["SessionID"] + list(feature_families)
)

check = session_features[
    ["SessionID"] + list(feature_families)
].merge(
    reference,
    on="SessionID",
    suffixes=("_jupyter", "_codex"),
    validate="one_to_one"
)

print("\nComparison with Codex:")

for feature in feature_families:

    actual = pd.to_numeric(
        check[f"{feature}_jupyter"],
        errors="coerce"
    )

    expected = pd.to_numeric(
        check[f"{feature}_codex"],
        errors="coerce"
    )

    matches = (
        actual.eq(expected).fillna(False)
        | (actual.isna() & expected.isna())
    )

    print(
        f"{feature}: "
        f"{int((~matches).sum())} mismatched sessions"
    )

In [ ]:
# Validate and export the completed Jupyter feature dataset

reference = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "session_features.csv"
)

feature_columns = [
    col for col in session_features.columns
    if col != "SessionID"
]

# Compare the updated 26 features across all sessions
comparison = session_features.merge(
    reference[["SessionID"] + feature_columns],
    on="SessionID",
    how="outer",
    suffixes=("_jupyter", "_codex"),
    validate="one_to_one",
    indicator=True
)

assert comparison["_merge"].eq("both").all()

mismatches = {}

for feature in feature_columns:
    actual = pd.to_numeric(
        comparison[f"{feature}_jupyter"],
        errors="coerce"
    )
    expected = pd.to_numeric(
        comparison[f"{feature}_codex"],
        errors="coerce"
    )

    matches = (
        actual.eq(expected).fillna(False)
        | (actual.isna() & expected.isna())
    )

    if (~matches).any():
        mismatches[feature] = int((~matches).sum())

print("Sessions:", len(session_features))
print("Features compared:", len(feature_columns))
print("Features matching exactly:", len(feature_columns) - len(mismatches))
print("Mismatches:", mismatches)

# Export only after successful comparison
if mismatches:
    raise ValueError("Feature differences remain. Export stopped.")

output_file = OUTPUT_DIR / "session_features_jupyter.csv"

session_features.to_csv(output_file, index=False)

print("Saved to:", output_file)